# Predicting Stellar Class
## Score: .96653


In [107]:
OVERNIGHT_MODE = False

In [108]:
import numpy as np
import pandas as pd

DATA_DIR = "playground-series-s6e6"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"train: {train.shape}    test: {test.shape}")
print()
print("dtypes:")
print(train.dtypes)
print()

print("class distribution (train):")
print(train["class"].value_counts(normalize=True).round(4).to_string())
print()

nulls = train.isna().sum()
print("nulls per column (train):")
print(nulls[nulls > 0].to_string() if nulls.any() else "  none")
print()

print("categorical cardinalities:")
for c in ["spectral_type", "galaxy_population"]:
    vals = sorted(train[c].unique().tolist())
    print(f"  {c}: {len(vals)} unique  ->  {vals}")
print()

print("numeric summary:")
train.drop(columns=["id"]).describe().T

train: (577347, 12)    test: (247435, 11)

dtypes:
id                     int64
alpha                float64
delta                float64
u                    float64
g                    float64
r                    float64
i                    float64
z                    float64
redshift             float64
spectral_type         object
galaxy_population     object
class                 object
dtype: object

class distribution (train):
class
GALAXY    0.6538
QSO       0.2029
STAR      0.1433

nulls per column (train):
  none

categorical cardinalities:
  spectral_type: 4 unique  ->  ['A/F', 'G/K', 'M', 'O/B']
  galaxy_population: 2 unique  ->  ['Blue_Cloud', 'Red_Sequence']

numeric summary:


,count,mean,std,min,25%,50%,75%,max
alpha,577347.0,181.616673,96.242941,0.011684,132.161499,188.681465,231.829693,359.999810
delta,577347.0,21.834654,18.933570,-17.966988,2.474097,21.484412,36.988310,79.158322
u,577347.0,22.441926,2.018135,-0.139225,20.977090,22.570222,23.869103,28.253263
g,577347.0,21.007273,1.795426,13.535483,19.865005,21.467820,22.292715,27.620208
r,577347.0,19.962811,1.648964,12.579407,18.820671,20.431153,21.164096,25.254499
i,577347.0,19.378911,1.580059,11.962781,18.306820,19.631642,20.608191,27.910853
z,577347.0,19.041136,1.584365,11.682803,17.973192,19.188598,20.162111,26.826867
redshift,577347.0,0.723135,0.810070,-0.009970,0.181052,0.497525,0.881390,7.010780


In [109]:
BASE_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift", "spectral_type", "galaxy_population"]
CAT_FEATURES = ["spectral_type", "galaxy_population"]
TARGET = "class"
LABELS = ["GALAXY", "QSO", "STAR"]
label_to_idx = {l: i for i, l in enumerate(LABELS)}
idx_to_label = {i: l for l, i in label_to_idx.items()}

MAGNITUDE_COLS = ["u", "g", "r", "i", "z"]
SENTINELS = {-9999.0, -999.0, -99.0, 99.0, 999.0, 9999.0}

def clean_sentinels(df):
    df = df.copy()
    cleaned = 0
    for c in MAGNITUDE_COLS + ["redshift"]:
        mask = df[c].isin(SENTINELS) | (df[c] > 50) | (df[c] < -50)
        if mask.any():
            cleaned += int(mask.sum())
            df.loc[mask, c] = np.nan
    return df, cleaned

def add_color_features(df, extended=False):
    df = df.copy()
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    df["u_r"] = df["u"] - df["r"]
    df["g_i"] = df["g"] - df["i"]
    df["u_z"] = df["u"] - df["z"]
    if extended:
        df["u_i"] = df["u"] - df["i"]
        df["g_z"] = df["g"] - df["z"]
        df["r_z"] = df["r"] - df["z"]
        df["mag_mean"] = df[MAGNITUDE_COLS].mean(axis=1)
        df["mag_std"] = df[MAGNITUDE_COLS].std(axis=1)
        df["log1p_redshift"] = np.sign(df["redshift"]) * np.log1p(np.abs(df["redshift"]))
        df["u_g_sq"] = df["u_g"] ** 2
        df["g_r_sq"] = df["g_r"] ** 2
    return df

train_fe, train_cleaned = clean_sentinels(train)
test_fe, test_cleaned = clean_sentinels(test)
print(f"sentinel cleaning -> train: {train_cleaned} cells nan'd    test: {test_cleaned} cells nan'd")

train_fe = add_color_features(train_fe, extended=OVERNIGHT_MODE)
test_fe = add_color_features(test_fe, extended=OVERNIGHT_MODE)

COLOR_FEATURES = ["u_g", "g_r", "r_i", "i_z", "u_r", "g_i", "u_z"]
EXTENDED_FEATURES = ["u_i", "g_z", "r_z", "mag_mean", "mag_std", "log1p_redshift", "u_g_sq", "g_r_sq"]
ALL_FEATURES = BASE_FEATURES + COLOR_FEATURES + (EXTENDED_FEATURES if OVERNIGHT_MODE else [])

for c in CAT_FEATURES:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = test_fe[c].astype("category")

X = train_fe[ALL_FEATURES]
y = train_fe[TARGET].map(label_to_idx).astype(int)
X_test = test_fe[ALL_FEATURES]

print(f"X: {X.shape}    y: {y.shape}    X_test: {X_test.shape}")
print(f"features ({len(ALL_FEATURES)}): {ALL_FEATURES}")

sentinel cleaning -> train: 0 cells nan'd    test: 0 cells nan'd
X: (577347, 17)    y: (577347,)    X_test: (247435, 17)
features (17): ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'u_z']


In [110]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

freq = np.bincount(y.to_numpy(), minlength=len(LABELS)) / len(y)
inv_freq = 1.0 / freq

WEIGHT_ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]

Xw_tr, Xw_va, yw_tr, yw_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

tune_params = {
    "objective": "multiclass",
    "num_class": len(LABELS),
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "verbose": -1,
    "seed": 42,
}

best_alpha, class_weight_vec, best_score = 1.0, None, -1.0
for alpha in WEIGHT_ALPHAS:
    vec = inv_freq ** alpha
    vec = vec / vec.mean()
    w = vec[yw_tr.to_numpy()]
    dtr = lgb.Dataset(Xw_tr, yw_tr, weight=w, categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(Xw_va, yw_va, categorical_feature=CAT_FEATURES, reference=dtr)
    m = lgb.train(tune_params, dtr, num_boost_round=2000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)])
    pred = m.predict(Xw_va, num_iteration=m.best_iteration)
    s = balanced_accuracy_score(yw_va, pred.argmax(axis=1))
    print(f"alpha={alpha:.2f}: balanced_acc = {s:.5f}    weights = {np.round(vec, 3)}")
    if s > best_score:
        best_score, best_alpha, class_weight_vec = s, alpha, vec

print(f"\nbest alpha: {best_alpha:.2f}    class weights: {np.round(class_weight_vec, 4)}    val balanced_acc: {best_score:.5f}")

alpha=0.00: balanced_acc = 0.95581    weights = [1. 1. 1.]
alpha=0.25: balanced_acc = 0.95900    weights = [0.789 1.057 1.153]
alpha=0.50: balanced_acc = 0.96171    weights = [0.608 1.092 1.3  ]
alpha=0.75: balanced_acc = 0.96269    weights = [0.46  1.105 1.435]
alpha=1.00: balanced_acc = 0.96355    weights = [0.341 1.1   1.558]
alpha=1.25: balanced_acc = 0.96401    weights = [0.25  1.081 1.669]
alpha=1.50: balanced_acc = 0.96392    weights = [0.181 1.05  1.769]

best alpha: 1.25    class weights: [0.2503 1.0806 1.6692]    val balanced_acc: 0.96401


In [111]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

NOISE_SEEDS = [42, 1337, 2024, 7, 2025]
NOISE_N_SPLITS = 5

noise_params = {
    "objective": "multiclass",
    "num_class": len(LABELS),
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "min_data_in_leaf": 200,
    "verbose": -1,
}

seed_oof_scores = []
fold_scores = []

for seed in NOISE_SEEDS:
    skf = StratifiedKFold(n_splits=NOISE_N_SPLITS, shuffle=True, random_state=seed)
    oof_noise = np.zeros((len(X), 3))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        w_tr = class_weight_vec[y_tr.to_numpy()]

        dtrain = lgb.Dataset(X_tr, y_tr, weight=w_tr, categorical_feature=CAT_FEATURES)
        dvalid = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtrain)

        model = lgb.train(
            {**noise_params, "seed": seed},
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )

        va_pred = model.predict(X_va, num_iteration=model.best_iteration)
        oof_noise[va_idx] = va_pred
        fs = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
        fold_scores.append(fs)

    oof_score = balanced_accuracy_score(y, oof_noise.argmax(axis=1))
    seed_oof_scores.append(oof_score)
    print(f"seed {seed}: OOF balanced_acc = {oof_score:.5f}")

seed_arr = np.array(seed_oof_scores)
fold_arr = np.array(fold_scores)
print()
print(f"seed OOF mean: {seed_arr.mean():.5f}    std: {seed_arr.std():.5f}    min: {seed_arr.min():.5f}    max: {seed_arr.max():.5f}")
print(f"fold  mean: {fold_arr.mean():.5f}    std: {fold_arr.std():.5f}")
print(f"approx noise floor (1 seed-std): {seed_arr.std():.5f}")
print(f"gap to 0.96731 (leader): {0.96731 - seed_arr.mean():.5f}")

seed 42: OOF balanced_acc = 0.96438
seed 1337: OOF balanced_acc = 0.96450
seed 2024: OOF balanced_acc = 0.96435
seed 7: OOF balanced_acc = 0.96410
seed 2025: OOF balanced_acc = 0.96423

seed OOF mean: 0.96431    std: 0.00014    min: 0.96410    max: 0.96450
fold  mean: 0.96431    std: 0.00060
approx noise floor (1 seed-std): 0.00014
gap to 0.96731 (leader): 0.00300


In [112]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

if OVERNIGHT_MODE:
    N_SPLITS = 15
    SEEDS = [42, 1337, 2024]
    LEARNING_RATE = 0.02
    NUM_BOOST_ROUND = 15000
    EARLY_STOPPING_ROUNDS = 200
else:
    N_SPLITS = 5
    SEEDS = [42]
    LEARNING_RATE = 0.05
    NUM_BOOST_ROUND = 5000
    EARLY_STOPPING_ROUNDS = 100

TOTAL_FITS = N_SPLITS * len(SEEDS)

base_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": LEARNING_RATE,
    "num_leaves": 63,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "min_data_in_leaf": 200,
    "verbose": -1,
}

oof_lgb = np.zeros((len(X), 3))
test_pred_lgb = np.zeros((len(X_test), 3))
fi_lgb = np.zeros(len(ALL_FEATURES))
fit_idx = 0

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros((len(X), 3))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        fit_idx += 1
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        w_tr = class_weight_vec[y_tr.to_numpy()]

        dtrain = lgb.Dataset(X_tr, y_tr, weight=w_tr, categorical_feature=CAT_FEATURES)
        dvalid = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtrain)

        params = {**base_params, "seed": seed}
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(0)],
        )

        va_pred = model.predict(X_va, num_iteration=model.best_iteration)
        seed_oof[va_idx] = va_pred
        test_pred_lgb += model.predict(X_test, num_iteration=model.best_iteration)
        fi_lgb += model.feature_importance(importance_type="gain")

        fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
        print(f"fold {fit_idx}/{TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {model.best_iteration}")

    oof_lgb += seed_oof

oof_lgb /= len(SEEDS)
test_pred_lgb /= TOTAL_FITS
fi_lgb /= TOTAL_FITS

lgb_oof_score = balanced_accuracy_score(y, oof_lgb.argmax(axis=1))
print(f"\nLightGBM OOF balanced_acc: {lgb_oof_score:.5f}")

fold 1/5: balanced_acc = 0.96469    best_iter = 1759
fold 2/5: balanced_acc = 0.96443    best_iter = 1705
fold 3/5: balanced_acc = 0.96423    best_iter = 1715
fold 4/5: balanced_acc = 0.96416    best_iter = 1615
fold 5/5: balanced_acc = 0.96440    best_iter = 1696

LightGBM OOF balanced_acc: 0.96438


In [113]:
oof_cb = None
test_pred_cb = None

if OVERNIGHT_MODE:
    try:
        from catboost import CatBoostClassifier
        _cb_available = True
    except ImportError:
        print("CatBoost not installed. Run: pip install catboost")
        print("Skipping CatBoost stage.")
        _cb_available = False

    if _cb_available:
        CB_N_SPLITS = 5
        CB_SEEDS = [42]
        CB_TOTAL_FITS = CB_N_SPLITS * len(CB_SEEDS)

        cat_idx = [ALL_FEATURES.index(c) for c in CAT_FEATURES]

        X_cb = X.copy()
        X_test_cb = X_test.copy()
        for c in CAT_FEATURES:
            X_cb[c] = X_cb[c].astype(str)
            X_test_cb[c] = X_test_cb[c].astype(str)

        oof_cb = np.zeros((len(X), 3))
        test_pred_cb = np.zeros((len(X_test), 3))
        fit_idx = 0

        for seed in CB_SEEDS:
            skf = StratifiedKFold(n_splits=CB_N_SPLITS, shuffle=True, random_state=seed)
            for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y), start=1):
                fit_idx += 1
                X_tr, X_va = X_cb.iloc[tr_idx], X_cb.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

                model = CatBoostClassifier(
                    iterations=15000,
                    learning_rate=0.03,
                    depth=7,
                    l2_leaf_reg=3.0,
                    loss_function="MultiClass",
                    eval_metric="MultiClass",
                    class_weights=list(class_weight_vec),
                    early_stopping_rounds=200,
                    thread_count=-1,
                    random_seed=seed,
                    verbose=0,
                )
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_idx)

                va_pred = model.predict_proba(X_va)
                oof_cb[va_idx] += va_pred / len(CB_SEEDS)
                test_pred_cb += model.predict_proba(X_test_cb) / CB_TOTAL_FITS

                fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
                print(f"cb fold {fit_idx}/{CB_TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {model.tree_count_}")

        cb_oof_score = balanced_accuracy_score(y, oof_cb.argmax(axis=1))
        print(f"\nCatBoost OOF balanced_acc: {cb_oof_score:.5f}")

In [114]:
oof_xgb = None
test_pred_xgb = None

try:
    import xgboost as xgb
    _xgb_available = True
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    print("Skipping XGBoost stage.")
    _xgb_available = False

if _xgb_available:
    if OVERNIGHT_MODE:
        XGB_N_SPLITS = 15
        XGB_SEEDS = [42, 1337, 2024]
        XGB_LR = 0.02
        XGB_NUM_ROUNDS = 15000
        XGB_EARLY_STOPPING = 200
    else:
        XGB_N_SPLITS = 5
        XGB_SEEDS = [42]
        XGB_LR = 0.05
        XGB_NUM_ROUNDS = 5000
        XGB_EARLY_STOPPING = 100

    XGB_TOTAL_FITS = XGB_N_SPLITS * len(XGB_SEEDS)

    xgb_params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "max_depth": 8,
        "learning_rate": XGB_LR,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 5,
        "reg_lambda": 1.0,
        "verbosity": 0,
    }

    dtest = xgb.DMatrix(X_test, enable_categorical=True)

    oof_xgb = np.zeros((len(X), 3))
    test_pred_xgb = np.zeros((len(X_test), 3))
    fit_idx = 0

    for seed in XGB_SEEDS:
        skf = StratifiedKFold(n_splits=XGB_N_SPLITS, shuffle=True, random_state=seed)
        seed_oof = np.zeros((len(X), 3))

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            fit_idx += 1
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            w_tr = class_weight_vec[y_tr.to_numpy()]

            dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, enable_categorical=True)
            dvalid = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)

            params = {**xgb_params, "seed": seed}
            model = xgb.train(
                params,
                dtrain,
                num_boost_round=XGB_NUM_ROUNDS,
                evals=[(dvalid, "valid")],
                early_stopping_rounds=XGB_EARLY_STOPPING,
                verbose_eval=False,
            )

            best_it = model.best_iteration
            va_pred = model.predict(dvalid, iteration_range=(0, best_it + 1))
            seed_oof[va_idx] = va_pred
            test_pred_xgb += model.predict(dtest, iteration_range=(0, best_it + 1))

            fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
            print(f"xgb fold {fit_idx}/{XGB_TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {best_it}")

        oof_xgb += seed_oof

    oof_xgb /= len(XGB_SEEDS)
    test_pred_xgb /= XGB_TOTAL_FITS

    xgb_oof_score = balanced_accuracy_score(y, oof_xgb.argmax(axis=1))
    print(f"\nXGBoost OOF balanced_acc: {xgb_oof_score:.5f}")

xgb fold 1/5: balanced_acc = 0.96418    best_iter = 2277
xgb fold 2/5: balanced_acc = 0.96413    best_iter = 2424
xgb fold 3/5: balanced_acc = 0.96346    best_iter = 2304
xgb fold 4/5: balanced_acc = 0.96386    best_iter = 2310
xgb fold 5/5: balanced_acc = 0.96371    best_iter = 2463

XGBoost OOF balanced_acc: 0.96387


In [115]:
from itertools import product

base_models = {"lgb": (oof_lgb, test_pred_lgb)}
if oof_cb is not None:
    base_models["cb"] = (oof_cb, test_pred_cb)
if oof_xgb is not None:
    base_models["xgb"] = (oof_xgb, test_pred_xgb)

model_names = list(base_models.keys())
oof_stack = [base_models[m][0] for m in model_names]
test_stack = [base_models[m][1] for m in model_names]

def simplex_weights(n, step=0.05):
    vals = np.round(np.arange(0, 1 + 1e-9, step), 4)
    if n == 1:
        return [(1.0,)]
    if n == 2:
        return [(a, round(1 - a, 4)) for a in vals]
    combos = []
    for a in vals:
        for b in vals:
            c = round(1 - a - b, 4)
            if c >= -1e-9:
                combos.append((a, b, c))
    return combos

best_weights = None
best_blend_score = -1.0
for w in simplex_weights(len(model_names)):
    blend = sum(wi * o for wi, o in zip(w, oof_stack))
    s = balanced_accuracy_score(y, blend.argmax(axis=1))
    if s > best_blend_score:
        best_blend_score, best_weights = s, w

oof_blend = sum(wi * o for wi, o in zip(best_weights, oof_stack))
test_pred_blend = sum(wi * t for wi, t in zip(best_weights, test_stack))
print("blend weights: " + "  ".join(f"{m}={w:.2f}" for m, w in zip(model_names, best_weights)))
print(f"blended OOF balanced_acc: {best_blend_score:.5f}")

def search_mult(probs, target, g1, g2, best):
    best_s1, best_s2, best_score = best
    for s1 in g1:
        scaled1 = probs[:, 1] * s1
        for s2 in g2:
            pred = np.column_stack([probs[:, 0], scaled1, probs[:, 2] * s2]).argmax(axis=1)
            s = balanced_accuracy_score(target, pred)
            if s > best_score:
                best_score, best_s1, best_s2 = s, s1, s2
    return best_s1, best_s2, best_score

base_scaled_score = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
coarse = np.linspace(0.2, 3.0, 29)
c1, c2, cs = search_mult(oof_blend, y, coarse, coarse, (1.0, 1.0, base_scaled_score))
fine1 = np.linspace(max(0.05, c1 - 0.2), c1 + 0.2, 41)
fine2 = np.linspace(max(0.05, c2 - 0.2), c2 + 0.2, 41)
best_s1, best_s2, best_scaled_score = search_mult(oof_blend, y, fine1, fine2, (c1, c2, cs))
best_scalars = (1.0, best_s1, best_s2)

scalars = np.array(best_scalars)
oof_final = oof_blend * scalars
test_pred_final = test_pred_blend * scalars

print(f"best class scalars: GALAXY={best_scalars[0]:.3f}  QSO={best_scalars[1]:.3f}  STAR={best_scalars[2]:.3f}")
print(f"OOF balanced_acc after scaling: {best_scaled_score:.5f}    (delta: {best_scaled_score - base_scaled_score:+.5f})")

blend weights: lgb=0.80  xgb=0.20
blended OOF balanced_acc: 0.96459
best class scalars: GALAXY=1.000  QSO=1.760  STAR=1.770
OOF balanced_acc after scaling: 0.96560    (delta: +0.00101)


In [116]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_final = oof_final.argmax(axis=1)

cm = confusion_matrix(y, y_pred_final)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print("confusion matrix (counts):")
print(cm_df.to_string())
print()

cm_norm = cm / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print("row-normalized (diagonal = per-class recall):")
print(cm_norm_df.round(5).to_string())
print()

per_class_recall = np.diag(cm_norm)
print("per-class recall (mean = balanced accuracy):")
for label, r in zip(LABELS, per_class_recall):
    print(f"  {label}: {r:.5f}")
print(f"  mean:        {per_class_recall.mean():.5f}")
print()

off_diag = []
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if i != j:
            off_diag.append((LABELS[i], LABELS[j], int(cm[i, j]), cm_norm[i, j]))
off_diag.sort(key=lambda t: -t[2])
print("top off-diagonal errors (true -> pred):")
for true_l, pred_l, count, rate in off_diag:
    print(f"  {true_l:>6} -> {pred_l:<6}  {count:>8}    ({rate:.4%} of true {true_l})")
print()

print("classification report:")
print(classification_report(y, y_pred_final, target_names=LABELS, digits=5))

if "fi_lgb" in globals():
    fi_df = pd.DataFrame({"feature": ALL_FEATURES, "gain": fi_lgb}).sort_values("gain", ascending=False).reset_index(drop=True)
    total_gain = fi_df["gain"].sum()
    fi_df["pct_of_total"] = (fi_df["gain"] / total_gain * 100).round(3)
    fi_df["cumulative_pct"] = fi_df["pct_of_total"].cumsum().round(3)
    print("LightGBM feature importance (gain, averaged across all folds):")
    print(fi_df.to_string(index=False))

    weak = fi_df[fi_df["pct_of_total"] < 0.5]["feature"].tolist()
    if weak:
        print(f"\nfeatures contributing <0.5% of total gain (drop candidates): {weak}")
else:
    print("feature importance not available -- rerun the LightGBM CV cell to populate fi_lgb")

confusion matrix (counts):
             pred_GALAXY  pred_QSO  pred_STAR
true_GALAXY       359933      5767      11780
true_QSO            1733    114307       1103
true_STAR           2224       465      80035

row-normalized (diagonal = per-class recall):
             pred_GALAXY  pred_QSO  pred_STAR
true_GALAXY      0.95352   0.01528    0.03121
true_QSO         0.01479   0.97579    0.00942
true_STAR        0.02688   0.00562    0.96749

per-class recall (mean = balanced accuracy):
  GALAXY: 0.95352
  QSO: 0.97579
  STAR: 0.96749
  mean:        0.96560

top off-diagonal errors (true -> pred):
  GALAXY -> STAR       11780    (3.1207% of true GALAXY)
  GALAXY -> QSO         5767    (1.5278% of true GALAXY)
    STAR -> GALAXY      2224    (2.6885% of true STAR)
     QSO -> GALAXY      1733    (1.4794% of true QSO)
     QSO -> STAR        1103    (0.9416% of true QSO)
    STAR -> QSO          465    (0.5621% of true STAR)

classification report:
              precision    recall  f1-score

In [117]:
pred_labels = np.array([idx_to_label[i] for i in test_pred_final.argmax(axis=1)])
submission = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission.csv", index=False)

print(f"submission shape: {submission.shape}")
print()
print("predicted class distribution:")
print(submission["class"].value_counts(normalize=True).round(4).to_string())
print()
submission.head()

submission shape: (247435, 2)

predicted class distribution:
class
GALAXY    0.6311
QSO       0.2086
STAR      0.1603



,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
